# NB-06 | End-to-End Pipeline Runner

Chains Steps 1–5 into a single execution by calling the logic from each notebook as importable functions. Set `TARGET_REPO` and run all cells.

**Key improvements over v1:**
- All shared logic (grounding validator, IAE scorer, retry helper) is imported from `pipeline_utils` — no inline duplication
- Step timing is computed correctly using `perf_counter` checkpoints, not cumulative subtraction
- `mitigation_limit` is an explicit, configurable parameter — not a silent `[:10]` truncation
- Outputs are namespaced by `run_id` so consecutive runs don't overwrite each other
- Run cost is estimated pre-flight and printed before any API calls are made

**Time target: under 90 seconds for repos with < 30 components.**

In [ ]:
!pip install -q openai anthropic tiktoken requests PyYAML

import os, sys
sys.path.insert(0, ".")

from pipeline_utils import (
    github_get, save_json, load_json, get_logger,
    filter_grounded_threats, compute_iae_score,
    parse_json_response, call_with_retry, sanitise_markdown,
)

import json, time, base64, yaml, datetime, re
from pathlib import Path
from collections import defaultdict, Counter
from time import perf_counter
from openai import OpenAI
import anthropic
import requests
import tiktoken

oai    = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
claude = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
log    = get_logger("NB06")

## Configuration

In [ ]:
TARGET_REPO       = os.environ.get("TARGET_REPO", "tiangolo/fastapi")
GITHUB_TOKEN     = os.environ.get("GITHUB_TOKEN", "")

# File ingestion
PRIORITY_EXTS    = {".py", ".ts", ".js", ".go", ".java", ".rb"}
PRIORITY_PATS    = ["route","controller","auth","model","handler",
                     "api","middleware","service","config","db"]
MAX_FILES        = 25
TOKEN_BUDGET     = 10_000
MAX_FILE_CHARS   = 3_000

# Mitigation generation
MITIG_LIMIT      = 15   # max HIGH/CRITICAL threats to generate mitigations for
BATCH_SIZE       = 5

GH_HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"} if GITHUB_TOKEN else {}
RUN_ID = (
    f"{TARGET_REPO.replace('/', '__')}"
    f"__{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"
)

log.info("Repo         : %s", TARGET_REPO)
log.info("Authenticated: %s", bool(GITHUB_TOKEN))
log.info("Run ID       : %s", RUN_ID)

## Step 1 — Ingest

In [ ]:
t_step = perf_counter()
times: dict[str, float] = {}

def file_priority(p: str) -> int:
    return sum(1 for x in PRIORITY_PATS if x in p.lower()) + (1 if Path(p).suffix in PRIORITY_EXTS else 0)


def fetch_tree(repo: str) -> list[dict]:
    resp = github_get(
        f"https://api.github.com/repos/{repo}/git/trees/HEAD?recursive=1",
        GH_HEADERS, logger=log
    )
    if resp is None:
        raise ValueError(f"Repo not found: {repo}")
    data = resp.json()
    if data.get("truncated"):
        log.warning("Tree truncated — large repo")
    return [i for i in data.get("tree", []) if i["type"] == "blob"]


def fetch_content(repo: str, path: str) -> str:
    resp = github_get(
        f"https://api.github.com/repos/{repo}/contents/{path}",
        GH_HEADERS, logger=log
    )
    if resp is None:
        return ""
    d = resp.json()
    if d.get("encoding") != "base64":
        return ""
    try:
        return base64.b64decode(d["content"]).decode("utf-8", "replace")[:MAX_FILE_CHARS]
    except Exception as exc:
        log.warning("Decode error for %s: %s", path, exc)
        return ""


tree      = fetch_tree(TARGET_REPO)
all_paths = {f["path"] for f in tree}

scored_files = sorted(
    [(f["path"], file_priority(f["path"]))
     for f in tree if Path(f["path"]).suffix in PRIORITY_EXTS],
    key=lambda x: -x[1]
)[:MAX_FILES]

files: dict[str, str] = {}
for path, _ in scored_files:
    content = fetch_content(TARGET_REPO, path)
    if content.strip():
        files[path] = content
    time.sleep(0.07)

openapi = None
for cand in ["openapi.yaml", "openapi.json", "swagger.yaml", "swagger.json"]:
    if cand in all_paths:
        raw = fetch_content(TARGET_REPO, cand)
        try:
            openapi = yaml.safe_load(raw) if cand.endswith(".yaml") else json.loads(raw)
            if isinstance(openapi, dict) and any(k in openapi for k in ("paths", "openapi", "swagger")):
                log.info("OpenAPI found: %s", cand)
                break
        except Exception:
            openapi = None

valid_paths: set[str] = set(files.keys())   # full relative paths
times["ingest"] = perf_counter() - t_step
log.info("Step 1 done — %d files | openapi=%s | %.1fs",
         len(files), openapi is not None, times["ingest"])

## Step 2 — Extract Components (GPT-4o)

In [ ]:
t_step = perf_counter()

def tok(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-4o").encode(text))


ctx_parts: list[str] = []
used = 0
for path, content in files.items():
    snippet = f"### FILE: {path}\n{content}\n"
    t = tok(snippet)
    if used + t > TOKEN_BUDGET:
        break
    ctx_parts.append(snippet)
    used += t
code_ctx = "\n".join(ctx_parts)

COMP_SYS = (
    "You are an AppSec architect. Extract the security architecture as JSON only. "
    "In 'files', use exact relative paths as shown in the ### FILE: headers.\n"
    '{"components":[{"id":"str","name":"str","type":"service|database|auth|external_api|queue|cache|frontend",'
    '"description":"str","files":["exact/relative/path"],"data_handled":["str"]}],'
    '"data_flows":[{"from":"id","to":"id","data":"str","protocol":"str","authenticated":true}],'
    '"trust_boundaries":[{"id":"str","name":"str","description":"str","components_inside":["id"],"entry_points":["str"]}],'
    '"external_actors":[{"id":"str","name":"str","trust_level":"untrusted|semi-trusted|trusted"}]}'
)

openapi_section = ""
if openapi:
    paths = list((openapi.get("paths") or {}).keys())[:20]
    openapi_section = "\n### OPENAPI ENDPOINTS\n" + "\n".join(paths)

def _gpt_comps(messages):
    resp = oai.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.1,
        response_format={"type": "json_object"},
    )
    return parse_json_response(resp.choices[0].message.content, context="NB06-comps")


components = call_with_retry(
    _gpt_comps,
    [{"role": "system", "content": COMP_SYS},
     {"role": "user",   "content": f"Analyze:\n{code_ctx}{openapi_section}\n\nJSON only."}],
    max_retries=3, logger=log,
)

times["components"] = perf_counter() - t_step
log.info("Step 2 done — %d components | %.1fs",
         len(components.get("components", [])), times["components"])

## Step 3 — STRIDE Analysis (Claude Sonnet)

In [ ]:
t_step = perf_counter()

STRIDE_SYS = """
You are a senior AppSec engineer. Apply STRIDE to each component.
Every threat MUST cite a specific file path and function in its evidence field.
Every threat MUST include explicit integer values (1, 2, or 3) in iae_factors.
Respond ONLY with valid JSON.

IAE factors (1=low, 2=medium, 3=high):
  data_sensitivity, privilege_level, system_criticality,
  access_vector, exploit_input_control, attack_complexity,
  endpoint_visibility, auth_barrier, data_flow_reach

{"threats":[{"stride_category":"S|T|R|I|D|E","title":"str","description":"str",
"evidence":"exact/path/file.ext — function() — behaviour","attack_vector":"str",
"affected_assets":["str"],
"iae_factors":{"data_sensitivity":2,"privilege_level":1,"system_criticality":3,
"access_vector":3,"exploit_input_control":2,"attack_complexity":2,
"endpoint_visibility":3,"auth_barrier":2,"data_flow_reach":2}}]}
"""

all_flows      = components.get("data_flows", [])
stride_results = {}
total_raw_s    = 0
total_kept_s   = 0

for comp in components.get("components", []):
    flows    = [f for f in all_flows if f["from"] == comp["id"] or f["to"] == comp["id"]]
    flow_txt = "\n".join(
        f"  - {f['data']} via {f['protocol']} (auth={f['authenticated']})" for f in flows
    ) or "  none"
    prompt = (
        f"Component: {comp['name']} ({comp['type']})\n"
        f"Files: {', '.join(comp.get('files', []))}\n"
        f"Flows:\n{flow_txt}\n\n"
        "Apply STRIDE. Include real file path in evidence and all iae_factors as integers. JSON only."
    )

    def _claude_stride(messages, _comp=comp):
        r = claude.messages.create(
            model="claude-sonnet-4-6", max_tokens=1800,
            system=STRIDE_SYS, messages=messages,
        )
        raw = parse_json_response(r.content[0].text, context=f"NB06-stride-{_comp['name']}")
        return raw.get("threats", [])

    try:
        raw_threats = call_with_retry(
            _claude_stride,
            [{"role": "user", "content": prompt}],
            max_retries=3, logger=log,
        )
    except RuntimeError as exc:
        log.error("Skipping %s: %s", comp["name"], exc)
        raw_threats = []

    total_raw_s += len(raw_threats)
    grounded     = filter_grounded_threats(raw_threats, valid_paths,
                                           label=comp["name"], logger=log)
    total_kept_s += len(grounded)
    stride_results[comp["id"]] = {
        "component_name": comp["name"],
        "component_type": comp.get("type", "service"),
        "threats":        grounded,
    }

times["stride"] = perf_counter() - t_step
log.info("Step 3 done — %d grounded threats (%d discarded) | %.1fs",
         total_kept_s, total_raw_s - total_kept_s, times["stride"])

## Step 4 — IAE Scoring

In [ ]:
t_step = perf_counter()

flat: list[dict] = []
for cid, d in stride_results.items():
    for t in d["threats"]:
        scored = dict(t)
        scored.update(compute_iae_score(t))
        scored["component_id"]   = cid
        scored["component_name"] = d["component_name"]
        scored["component_type"] = d.get("component_type", "service")
        flat.append(scored)

flat.sort(key=lambda x: -x["total_raw"])
dist = Counter(t["severity"] for t in flat)

times["scoring"] = perf_counter() - t_step
log.info("Step 4 done — CRITICAL:%d HIGH:%d MEDIUM:%d LOW:%d | %.1fs",
         dist["CRITICAL"], dist["HIGH"], dist["MEDIUM"], dist["LOW"], times["scoring"])

## Step 5 — Mitigations & Report

In [ ]:
t_step = perf_counter()

MITIG_SYS = """
You are a senior AppSec engineer. Write actionable mitigations mapped to OWASP/NIST. JSON only.
{"mitigations":[{"threat_id":"component_id::threat_title","threat_title":"str",
"owasp_refs":["str"],"nist_refs":["str"],
"actions":["step1","step2","step3"],
"code_pattern":"str or null","code_language":"python|javascript|go|java|generic",
"effort":"low|medium|high"}]}
"""

priority = filter_grounded_threats(
    [t for t in flat if t["severity"] in ("CRITICAL", "HIGH")][:MITIG_LIMIT],
    valid_paths, label="mitig", logger=log,
)
log.info("Generating mitigations for %d threats (limit=%d)", len(priority), MITIG_LIMIT)

mitigations: list[dict] = []
for i in range(0, len(priority), BATCH_SIZE):
    batch = priority[i:i + BATCH_SIZE]
    prompt = "Generate mitigations:\n\n" + "\n\n".join(
        f"Threat ID  : {t.get('component_id','?')}::{t['title']}\n"
        f"Title      : {t['title']}\n"
        f"Component  : {t['component_name']} | Location: {t.get('threat_location','N/A')}\n"
        f"Score      : {t['final_score']}/10 ({t['severity']})\n"
        f"Description: {t['description']}"
        for t in batch
    ) + "\n\nJSON only."

    def _claude_mitig(messages):
        r = claude.messages.create(
            model="claude-sonnet-4-6", max_tokens=3000,
            system=MITIG_SYS, messages=messages,
        )
        raw = parse_json_response(r.content[0].text, context="NB06-mitig")
        return raw.get("mitigations", [])

    try:
        mitigations.extend(call_with_retry(
            _claude_mitig,
            [{"role": "user", "content": prompt}],
            max_retries=3, logger=log,
        ))
    except RuntimeError as exc:
        log.error("Mitigation batch %d failed: %s", i // BATCH_SIZE + 1, exc)

mitig_map: dict[str, dict] = {m["threat_id"]: m for m in mitigations}

# ── Assemble report ───────────────────────────────────────────────────────────
today = datetime.date.today().isoformat()
SEV_BADGE = {"CRITICAL": "🔴 CRITICAL", "HIGH": "🟠 HIGH",
             "MEDIUM": "🟡 MEDIUM", "LOW": "🟢 LOW"}

lines: list[str] = [
    f"# Threat Model — `{TARGET_REPO}`",
    "",
    f"> **Generated:** {today}  |  **Run ID:** `{RUN_ID}`",
    "> **Methodology:** STRIDE  |  **Scoring:** IAE (structured factors)",
    "", "---", "",
    "## Summary",
    "| Severity | Count |",
    "|----------|------:|",
    f"| CRITICAL | {dist['CRITICAL']} |",
    f"| HIGH     | {dist['HIGH']} |",
    f"| MEDIUM   | {dist['MEDIUM']} |",
    f"| LOW      | {dist['LOW']} |",
    f"| Total    | {len(flat)} |",
    "", "---", "", "## Findings", "",
]

for i, t in enumerate(flat, 1):
    m        = mitig_map.get(f"{t.get('component_id', '')}::{t['title']}", {})
    location = t.get("threat_location") or "unknown"
    lines += [
        f"### {i}. {SEV_BADGE.get(t['severity'], t['severity'])} "
        f"{sanitise_markdown(t['title'])}",
        f"**Component:** {sanitise_markdown(t['component_name'])} | "
        f"**STRIDE:** {t.get('stride_category','?')} | "
        f"**Score:** {t['final_score']}/10 | "
        f"**Location:** `{sanitise_markdown(location)}`",
        f"**IAE:** Impact {t.get('impact_score','?')}/9 | "
        f"Exploit {t.get('exploit_score','?')}/9 | "
        f"Exposure {t.get('exposure_score','?')}/9",
        "",
        f"**Description:** {sanitise_markdown(t['description'])}",
        "",
        f"**Evidence:** `{sanitise_markdown(t.get('evidence','N/A'))}`",
        "",
    ]
    if m:
        lines.append("**Mitigations:**")
        for a in m.get("actions", []):
            lines.append(f"- {sanitise_markdown(a)}")
        refs = m.get("owasp_refs", []) + m.get("nist_refs", [])
        if refs:
            lines += ["", f"**Controls:** {', '.join(refs)}"]
        code = m.get("code_pattern")
        lang = m.get("code_language", "python")
        if code:
            lines += ["", "**Code Pattern:**", f"```{lang}", code, "```"]
    lines += ["", "---", ""]

report = "\n".join(lines)
output_md = f"threat_model_report__{RUN_ID}.md"
with open(output_md, "w") as f:
    f.write(report)

times["report"] = perf_counter() - t_step
total_elapsed   = sum(times.values())

log.info("Report saved: %s", output_md)
print(f"\n=== PIPELINE COMPLETE ===")
print(f"Total elapsed : {total_elapsed:.1f}s")
for step, secs in times.items():
    print(f"  {step:12s}: {secs:.1f}s")
print(f"\nThreats: {len(flat)} | Mitigations: {len(mitigations)} | Run: {RUN_ID}")
print("\nTop 5 threats:")
for t in flat[:5]:
    print(f"  {t['final_score']:4.1f}  [{t['severity']:8}]  "
          f"{t['component_name'][:20]:20}  {t['title']}")